# Notebook 5. Mejoras encadenadas



Objetivo: añadir mejoras una sobre otra y ver en la tabla qué aporta cada una y cuánto cuesta en latencia y dinero.

- **V4**: V3 + reranker (`BAAI/bge-reranker-v2-m3`) sobre los 50 primeros candidatos.
- **V5**: V4 + enriquecimiento de metadatos (ruta de sección, cabecera de contexto y tabla serializada con su título).
- **V6**: V5 + transformación de la consulta (multi-query o HyDE).

Referencias:
- Retrieve & Re-Rank (sentence-transformers): https://sbert.net/examples/applications/retrieve_rerank/README.html
- Contextual Retrieval: https://anthropic.com/news/contextual-retrieval
- HyDE: https://arxiv.org/abs/2212.10496

In [1]:
from rag_lab import CFG, set_seeds, summary

# CFG.FAST_MODE = False  # corpus de control completo (por defecto se lee FAST_MODE de .env)
# CFG.DOC_PATH = CFG.DATA_DIR / "mi_documento.pdf"  # cualquier PDF sirve
set_seeds()
summary()

Documento:   /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/documento.pdf (existe)
FAST_MODE:   True
Dispositivo: cuda
GEN_MODEL:   google/gemini-3.1-flash-lite
JUDGE_MODEL: openai/gpt-5-mini
VLM_MODEL:   google/gemini-3.1-flash-lite
OpenRouter:  clave configurada


Trazas:      jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl  (alumno: profesor)


/home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/rag_lab/tracing.py:49: UserWarning: No se pudo conectar con Langfuse ([Errno 111] Connection refused): las trazas se guardan en JSONL.
  warnings.warn(f"No se pudo conectar con Langfuse ({e}): las trazas se guardan en JSONL.")


In [2]:
import pandas as pd
from rag_lab import evaluation, models, results, tracing, versions

questions = evaluation.load_questions()
COLUMNS = ["tecnica", "recall@5", "MRR@10", "recall@5_visual", "fidelidad", "latencia_p50_ms",
            "latencia_p95_ms", "tokens", "coste_usd"]

## 1. V4: reranker

El recuperador (bi-encoder y BM25) codifica consulta y documento por separado: es rápido pero grueso.
Un cross-encoder lee consulta y fragmento juntos y puntúa su relación: es más preciso y mucho más lento,
así que solo reordena los 50 primeros candidatos del híbrido.

In [3]:
v3 = versions.build("V3")
v4 = versions.build("V4")
question = next(p for p in questions if p["tipo"] == "tabla" and p.get("tabla_gemela"))
print("Pregunta:", question["pregunta"], "| página correcta:", question["paginas"])
print("V3 (RRF):     ", [u["paginas"] for u in v3.retrieve(question["pregunta"])[:5]])
reranked = v4.retrieve(question["pregunta"])[:5]
print("V4 (reranker):", [u["paginas"] for u in reranked])
print("Puntuaciones del reranker:", [round(u["score_rerank"], 3) for u in reranked])

Pregunta: ¿Qué instrumento se utiliza para medir el impacto de las prácticas en la eficiencia y sostenibilidad en la tabla de indicadores de prácticas de economía circular? | página correcta: [18]
V3 (RRF):      [[17, 18], [17, 18], [8], [17], [19]]


V4 (reranker): [[17, 18], [17, 18], [8], [19], [1, 2]]
Puntuaciones del reranker: [0.944, 0.605, 0.047, 0.01, 0.009]


In [4]:
eval_v4 = evaluation.evaluate_rag(v4, questions)
results.table("pdf").loc[["V3", "V4"], COLUMNS].round(3)

,tecnica,recall@5,MRR@10,recall@5_visual,fidelidad,latencia_p50_ms,latencia_p95_ms,tokens,coste_usd
version,,,,,,,,,
V3,Híbrido BM25 + denso con RRF,1.0,0.858,1.000,0.8,637.565,815.302,33853.0,0.009
V4,V3 + reranker bge-reranker-v2-m3 (top-50),0.9,0.798,0.667,0.8,1907.240,2482.916,34403.0,0.009


Comparamos pregunta a pregunta la posición de la primera página correcta en V3 y en V4.
Fíjate en las preguntas visuales: el reranker solo lee texto, y algunas páginas visuales casi no tienen.

In [5]:
def positions(detail):
    return [next((i for i, u in enumerate(r, 1) if set(u) & set(c)), None)
            for r, c in zip(detail["recuperadas"], detail["correctas"])]

eval_v3 = evaluation.evaluate_rag(v3, questions, register=False)
comparison = pd.DataFrame({"id": eval_v3["detalle"]["id"], "tipo": eval_v3["detalle"]["tipo"],
                            "posición V3": positions(eval_v3["detalle"]), "posición V4": positions(eval_v4["detalle"])})
comparison[comparison["posición V3"] != comparison["posición V4"]]

,id,tipo,posición V3,posición V4
1,q02,texto,2,1.0
12,q13,tabla,2,5.0
14,q15,visual,3,NaN
16,q17,visual,2,4.0
18,q19,visual,3,NaN
19,q20,visual,1,2.0


## 2. V5: enriquecimiento de metadatos

Cada fragmento lleva delante una cabecera con el documento, su ruta de sección y sus páginas, y las tablas se
serializan como líneas `columna: valor` con el título de la tabla. Es la idea de Contextual Retrieval
(situar cada fragmento en su documento), aquí sin LLM: con la estructura que ya tenemos. Ayuda a distinguir
tablas casi idénticas, que solo se diferencian por su título.

In [6]:
structured = versions.pdf_fragments("cabeceras")
enriched = versions.enrich(structured, versions.document_title(versions.pdf.clean_pages()))
table = next(i for i, f in enumerate(structured) if f.get("tipo") == "tabla" and "LÍNEA 3" in f["texto"])
print("=== Original\n", structured[table]["texto"][:600])
print("\n=== Enriquecido\n", enriched[table]["texto"][:900])

=== Original
 ## 4.2 TAREAS POR LÍNEA Y FASE

|        | LÍNEA 1    | LÍNEA 2        | LÍNEA 3           | LÍNEA 4       | LÍNEA 5       | LÍNEA6        |
|--------|------------|----------------|-------------------|---------------|---------------|---------------|
| FASE 1 | T1, T2, T3 |                | T9, T10, T11, T12 |               |               |               |
| FASE 2 | T4         | T7             |                   |               |               | T24           |
| FASE 3 |            | T5, T6, T7, T8 |                   |               |               |               |
| FASE 4 |            |

=== Enriquecido
 Documento: Demostrador de un sistema de integración continúa desplegado bajo tecnologías Cloud para proyectos de Inteligencia Artificial | Sección: 4. DESARROLLO Y PLAN DE TRABAJO > 4.2 TAREAS POR LÍNEA Y FASE | Páginas: 12
Tabla: 4.2 TAREAS POR LÍNEA Y FASE
FASE 1 | LÍNEA 1: T1, T2, T3; LÍNEA 3: T9, T10, T11, T12
FASE 2 | LÍNEA 1: T4; LÍNEA 2: T7; LÍNEA6: T24
FASE

In [7]:
v5 = versions.build("V5")
evaluation.evaluate_rag(v5, questions)
results.table("pdf").loc[["V4", "V5"], COLUMNS].round(3)

,tecnica,recall@5,MRR@10,recall@5_visual,fidelidad,latencia_p50_ms,latencia_p95_ms,tokens,coste_usd
version,,,,,,,,,
V4,V3 + reranker bge-reranker-v2-m3 (top-50),0.9,0.798,0.667,0.8,1907.240,2482.916,34403.0,0.009
V5,"V4 + metadatos (ruta de sección, cabecera de c...",0.9,0.838,0.667,0.8,2366.005,3253.320,60631.0,0.016


## 3. V6: transformación de la consulta

- **Multi-query**: el LLM reformula la pregunta de 3 maneras; se busca con todas y se fusiona con RRF.
- **HyDE**: el LLM escribe un párrafo hipotético que respondería; se busca con su embedding.

Probamos las dos sobre V5. Regla de elección, fijada antes de medir: mejor MRR@10; si empatan, mayor
fidelidad; si siguen empatadas, multi-query, porque no introduce contenido inventado en la búsqueda
(HyDE puede escribir cifras que no existen y acercar la consulta a la tabla equivocada).

In [8]:
mq = versions.build("V6", v6_mode="multiquery")
hyde = versions.build("V6", v6_mode="hyde")
mq.version, hyde.version = "V6-multiquery", "V6-hyde"
print("Reformulaciones:", mq.prepare(question["pregunta"]))
print("\nDocumento hipotético de HyDE:\n", hyde.prepare(question["pregunta"]))

Reformulaciones: ['¿Qué herramienta de medición se emplea para evaluar el impacto de las prácticas en la eficiencia y sostenibilidad dentro de la tabla de indicadores de prácticas de economía circular?', '¿Cuál es el instrumento de evaluación definido en la tabla de indicadores de prácticas de economía circular para cuantificar la eficiencia y sostenibilidad de las prácticas?', '¿Qué mecanismo de control se utiliza en la tabla de indicadores de prácticas de economía circular para determinar el impacto de las prácticas en términos de eficiencia y sostenibilidad?']

Documento hipotético de HyDE:
 Para la evaluación del desempeño en la tabla de indicadores de prácticas de economía circular, se emplea el **Índice de Eficiencia y Sostenibilidad (IES)**. Este instrumento permite cuantificar de manera estandarizada el impacto de las prácticas implementadas, facilitando el seguimiento periódico de la optimización de recursos, la reducción de residuos y la mejora en la circularidad de los proce

In [9]:
eval_mq = evaluation.evaluate_rag(mq, questions, register=False)["resumen"]
eval_hyde = evaluation.evaluate_rag(hyde, questions, register=False)["resumen"]
options = pd.DataFrame([eval_mq, eval_hyde], index=["multiquery", "hyde"])
display(options[["recall@5", "MRR@10", "fidelidad", "latencia_p50_ms", "tokens", "coste_usd"]].round(4))

order = options.assign(prioridad=[1, 0]).sort_values(["MRR@10", "fidelidad", "prioridad"], ascending=False)
chosen = order.index[0]
print("Elegida para V6:", chosen)

,recall@5,MRR@10,fidelidad,latencia_p50_ms,tokens,coste_usd
multiquery,0.9,0.8375,0.8,3237.005,64087,0.0187
hyde,0.9,0.8375,0.8,3226.675,63405,0.0185


Elegida para V6: multiquery


In [10]:
v6 = mq if chosen == "multiquery" else hyde
v6.version = "V6"
evaluation.evaluate_rag(v6, questions)
results.table("pdf").loc[["V3", "V4", "V5", "V6"], COLUMNS].round(3)

,tecnica,recall@5,MRR@10,recall@5_visual,fidelidad,latencia_p50_ms,latencia_p95_ms,tokens,coste_usd
version,,,,,,,,,
V3,Híbrido BM25 + denso con RRF,1.0,0.858,1.000,0.8,637.565,815.302,33853.0,0.009
V4,V3 + reranker bge-reranker-v2-m3 (top-50),0.9,0.798,0.667,0.8,1907.240,2482.916,34403.0,0.009
V5,"V4 + metadatos (ruta de sección, cabecera de c...",0.9,0.838,0.667,0.8,2366.005,3253.320,60631.0,0.016
V6,"V5 + multi-query (3 reformulaciones, RRF)",0.9,0.838,0.667,0.8,3137.915,3951.608,64087.0,0.019


## 4. Dónde se va el tiempo

Las trazas guardan la latencia de cada paso. Para las llamadas que vienen de la caché se usa la latencia
de la llamada original, de modo que las cifras no dependen de si la caché estaba llena.

In [11]:
events = tracing.events_df()
events = events[events["version"].isin(["V3", "V4", "V5", "V6-multiquery", "V6-hyde"])]
events.pivot_table(index="version", columns="paso", values="latencia_ms", aggfunc="median").round(1)

paso,generate,query_transform,rerank,retrieve
version,,,,
V3,633.3,NaN,NaN,4.0
V4,732.2,NaN,1160.8,6.3
V5,765.0,NaN,1536.4,6.5
V6-hyde,850.3,910.4,1427.2,12.6
V6-multiquery,765.0,850.4,1510.3,20.7


In [12]:
# TODO: el reranker es el paso más lento. Mide recall@5, MRR@10 y la latencia p50 de V4 reordenando
# solo 10, 20 y 50 candidatos (parámetro n_candidates de versions.Hybrid).
for n in [10, 20, 50]:
    rag = None  # Completa: versions.Hybrid(structured, "cabeceras", version=f"V4-n{n}", rerank=True, n_candidates=n)
    if rag is None:
        print("Completa el TODO")
        break
    r = evaluation.evaluate_rag(rag, questions, with_judge=False, register=False)["resumen"]
    print(n, r["recall@5"], round(r["MRR@10"], 3), round(r["latencia_p50_ms"]))

Completa el TODO


In [13]:
models.release()

## Qué hemos visto

- Cada mejora tiene un coste: el reranker multiplica la latencia y la transformación de la consulta añade una llamada al LLM.
- El enriquecimiento con la estructura del documento no cuesta llamadas al LLM, pero alarga el texto que leen el reranker y el generador.
- Con 20 preguntas, varias versiones empatan: hay que decidir con la tabla completa (calidad, latencia y coste), no con una sola métrica.

## Ejercicio propuesto

Sustituye la cabecera de contexto de V5 por una frase de contexto generada con el LLM para cada fragmento
(como en Contextual Retrieval) y compara calidad y coste con V5.